In [ ]:
"""
Примеры кода "LCEL — Продвинутые паттерны"

Этот модуль демонстрирует:
1. Динамическая маршрутизация (RunnableBranch)
2. Устойчивость к сбоям (retry, fallbacks)
3. Стриминг и потоковая передача данных
4. Батчинг и конкурентность
5. Runtime-конфигурация
6. Callbacks и наблюдаемость
7. Кэширование
"""

In [ ]:

import os
import asyncio
from typing import Any, Dict, List
from dotenv import load_dotenv


In [2]:
load_dotenv()

True

# ============================================================================
# ЧАСТЬ 1: ДИНАМИЧЕСКАЯ МАРШРУТИЗАЦИЯ
# ============================================================================

In [3]:
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnablePassthrough,
    RunnableParallel,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI


In [4]:
"""
RunnableBranch — условное ветвление на основе проверки условий.

Проверяет условия по порядку и выполняет первый подходящий обработчик.
"""
print("=" * 60)
print("ПРИМЕР 1: RunnableBranch — условное ветвление")
print("=" * 60)

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)

# Специализированные цепочки для разных типов запросов
math_chain = (
    ChatPromptTemplate.from_template("Ты — математик. Реши задачу пошагово: {input}") | model | StrOutputParser()
)

code_chain = ChatPromptTemplate.from_template("Ты — программист. Напиши код для: {input}") | model | StrOutputParser()

general_chain = ChatPromptTemplate.from_template("Ответь на вопрос: {input}") | model | StrOutputParser()


# Условия для маршрутизации
def is_math(x: Dict) -> bool:
    keywords = ["вычисли", "посчитай", "сколько", "реши", "+", "-", "*", "/", "="]
    text = x.get("input", "").lower()
    return any(kw in text for kw in keywords)


def is_code(x: Dict) -> bool:
    keywords = ["код", "программа", "функция", "напиши", "python", "javascript"]
    text = x.get("input", "").lower()
    return any(kw in text for kw in keywords)


# Создаём маршрутизатор
router = RunnableBranch(
    (is_math, math_chain),
    (is_code, code_chain),
    general_chain,  # default
)

# Тестируем
test_cases = [
    {"input": "Сколько будет 15 * 7?"},
    {"input": "Напиши функцию для сортировки списка"},
    {"input": "Какая столица Франции?"},
]

print("\n1.1. Маршрутизация запросов:")
for case in test_cases:
    print(f"\n  Вопрос: '{case['input']}'")
    result = router.invoke(case)
    print(f"  Ответ: {result[:100]}...")

ПРИМЕР 1: RunnableBranch — условное ветвление

1.1. Маршрутизация запросов:

  Вопрос: 'Сколько будет 15 * 7?'
  Ответ: Чтобы решить задачу 15 * 7, можно воспользоваться пошаговым методом:

1. **Разделим 15 на 10 и 5**: ...

  Вопрос: 'Напиши функцию для сортировки списка'
  Ответ: Конечно! Вот пример функции на Python, которая сортирует список с использованием встроенной функции ...

  Вопрос: 'Какая столица Франции?'
  Ответ: Столицей Франции является Париж....


In [5]:
"""
Семантическая маршрутизация — классификация через LLM.

Модель сама определяет категорию запроса.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 2: Семантическая маршрутизация")
print("=" * 60)

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)

# Классификатор
classification_prompt = ChatPromptTemplate.from_template(
    """Классифицируй запрос пользователя.
Категории: technical, billing, complaint, general

Запрос: {input}

Ответь только названием категории одним словом:"""
)

classifier = classification_prompt | model | StrOutputParser()

# Специализированные обработчики
handlers = {
    "technical": ChatPromptTemplate.from_template("Ты — техподдержка. Помоги с проблемой: {input}")
    | model
    | StrOutputParser(),
    "billing": ChatPromptTemplate.from_template("Ты — специалист по оплате. Помоги с вопросом: {input}")
    | model
    | StrOutputParser(),
    "complaint": ChatPromptTemplate.from_template("Ты — менеджер по работе с жалобами. Ответь вежливо: {input}")
    | model
    | StrOutputParser(),
    "general": ChatPromptTemplate.from_template("Ответь на общий вопрос: {input}") | model | StrOutputParser(),
}


# Полная цепочка
def route_to_handler(x: Dict) -> str:
    category = x["category"].strip().lower()
    handler = handlers.get(category, handlers["general"])
    return handler.invoke({"input": x["input"]})


chain = RunnablePassthrough.assign(category=classifier) | RunnableLambda(route_to_handler)

test_queries = [
    "Не могу войти в аккаунт, постоянно ошибка",
    "Почему мне пришёл счёт на 1000 рублей?",
    "Это безобразие! Третий раз заказ опаздывает!",
    "Какие языки программирования вы поддерживаете?",
]

print("\n2.1. Классификация и маршрутизация:")
for query in test_queries:
    result = chain.invoke({"input": query})
    print(f"\n  Запрос: '{query[:40]}...'")
    print(f"  Ответ: {result[:80]}...")



ПРИМЕР 2: Семантическая маршрутизация

2.1. Классификация и маршрутизация:

  Запрос: 'Не могу войти в аккаунт, постоянно ошибк...'
  Ответ: Здравствуйте! Давайте попробуем разобраться с вашей проблемой. Пожалуйста, уточн...

  Запрос: 'Почему мне пришёл счёт на 1000 рублей?...'
  Ответ: Чтобы понять, почему вам пришёл счёт на 1000 рублей, нужно рассмотреть несколько...

  Запрос: 'Это безобразие! Третий раз заказ опаздыв...'
  Ответ: Уважаемый(ая) [Имя клиента],

Благодарю вас за ваше сообщение и приношу искренни...

  Запрос: 'Какие языки программирования вы поддержи...'
  Ответ: Мы поддерживаем множество языков программирования, включая, но не ограничиваясь:...


# ============================================================================
# ЧАСТЬ 2: УСТОЙЧИВОСТЬ К СБОЯМ
# ============================================================================

In [7]:
"""
with_retry() — автоматические повторные попытки при ошибках.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 3: with_retry() — повторные попытки")
print("=" * 60)

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)

# Базовая модель с retry
robust_model = model.with_retry(
    stop_after_attempt=3,  # Максимум 3 попытки
    wait_exponential_jitter=True,  # Экспоненциальная задержка со случайностью
)

print("\n3.1. Модель с retry:")
print("  stop_after_attempt=3")
print("  wait_exponential_jitter=True")

# Демонстрация использования
prompt = ChatPromptTemplate.from_template("Скажи '{word}' на японском")
chain = prompt | robust_model | StrOutputParser()

result = chain.invoke({"word": "привет"})
print(f"\n  Результат: {result}")


ПРИМЕР 3: with_retry() — повторные попытки

3.1. Модель с retry:
  stop_after_attempt=3
  wait_exponential_jitter=True

  Результат: Привет на японском языке будет "こんにちは" (конничива).


In [9]:
"""
with_fallbacks() — резервные варианты при сбоях.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 4: with_fallbacks() — резервные варианты")
print("=" * 60)

# Основная модель
primary = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)

# Резервная модель (в реальности это может быть другой провайдер)
backup = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0.5,
)

# Модель с fallback
resilient_model = primary.with_fallbacks([backup])

print("\n4.1. Структура fallback:")
print("  Primary: gpt-4o-mini (temperature=0)")
print("  Backup: gpt-4o-mini (temperature=0.5)")

prompt = ChatPromptTemplate.from_template("Объясни {topic} простыми словами")
chain = prompt | resilient_model | StrOutputParser()

result = chain.invoke({"topic": "квантовые вычисления"})
print(f"\n  Результат: {result[:100]}...")


ПРИМЕР 4: with_fallbacks() — резервные варианты

4.1. Структура fallback:
  Primary: gpt-4o-mini (temperature=0)
  Backup: gpt-4o-mini (temperature=0.5)

  Результат: Квантовые вычисления — это новый способ обработки информации, который использует принципы квантовой ...


In [11]:
"""
Комбинирование retry и fallbacks для максимальной устойчивости.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 5: Комбинирование retry + fallbacks")
print("=" * 60)

primary = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)
backup = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0.3,
)

# Каждая модель с retry + fallback между ними
resilient = primary.with_retry(stop_after_attempt=2).with_fallbacks([backup.with_retry(stop_after_attempt=2)])

print("\n5.1. Структура устойчивой цепочки:")
print("  1. Primary (retry x2)")
print("  2. Backup (retry x2)")
print("  Итого до 4 попыток")

prompt = ChatPromptTemplate.from_template("Что такое {concept}?")
chain = prompt | resilient | StrOutputParser()

result = chain.invoke({"concept": "machine learning"})
print(f"\n  Результат: {result[:100]}...")


ПРИМЕР 5: Комбинирование retry + fallbacks

5.1. Структура устойчивой цепочки:
  1. Primary (retry x2)
  2. Backup (retry x2)
  Итого до 4 попыток

  Результат: Машинное обучение (machine learning) — это область искусственного интеллекта, которая занимается раз...


In [12]:
"""
Кастомная обработка ошибок в цепочках.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 6: Кастомная обработка ошибок")
print("=" * 60)


from langchain_core.runnables import chain


model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)
parser = StrOutputParser()


@chain
def safe_invoke(input_data: Dict) -> Dict:
    """Безопасный вызов с обработкой ошибок."""
    try:
        prompt = ChatPromptTemplate.from_template("Ответь на: {question}")
        result = (prompt | model | parser).invoke(input_data)
        return {"success": True, "result": result}
    except Exception as e:
        return {"success": False, "error": str(e), "result": None}


print("\n6.1. Безопасный вызов:")
result = safe_invoke.invoke({"question": "Что такое Python?"})
print(f"  Success: {result['success']}")
print(f"  Result: {result['result'][:80] if result['result'] else 'N/A'}...")



ПРИМЕР 6: Кастомная обработка ошибок

6.1. Безопасный вызов:
  Success: True
  Result: Python — это высокоуровневый язык программирования, который был создан в конце 1...


# ============================================================================
# ЧАСТЬ 3: СТРИМИНГ
# ============================================================================

In [14]:
"""
Базовый стриминг с методом stream().
"""
print("\n" + "=" * 60)
print("ПРИМЕР 7: Базовый стриминг")
print("=" * 60)

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0.7,
)
prompt = ChatPromptTemplate.from_template("Напиши короткое стихотворение о {topic}")
chain = prompt | model | StrOutputParser()

print("\n7.1. Потоковый вывод:")
print("  ", end="")
for chunk in chain.stream({"topic": "программировании"}):
    print(chunk, end="", flush=True)
print()


ПРИМЕР 7: Базовый стриминг

7.1. Потоковый вывод:
  В коде строки, как ноты в музыке,  
Логика танцует, решая задачи.  
Цифры и буквы — в едином ритме,  
Создаём миры, где нет ничего иначе.

Алгоритмы, как звёзды, сверкают,  
В каждом решении — тайна и смысл.  
Программирование — искусство, что знает,  
Как из простого создать сложный мир.


In [15]:
"""
Расширенный стриминг с astream_events().
"""
print("\n" + "=" * 60)
print("ПРИМЕР 8: Стриминг событий")
print("=" * 60)


async def stream_with_events():
    model = ChatOpenAI(
        base_url=os.getenv("OPENROUTER_BASE_URL"),
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model="gpt-4o-mini",
        temperature=0,
    )
    prompt = ChatPromptTemplate.from_template("Объясни {concept} в 2 предложениях")
    chain = prompt | model | StrOutputParser()

    print("\n8.1. События выполнения:")
    async for event in chain.astream_events({"concept": "рекурсия"}, version="v2"):
        kind = event["event"]
        if kind == "on_chat_model_start":
            print(f"  [START] Модель начала генерацию")
        elif kind == "on_chat_model_stream":
            content = event["data"]["chunk"].content
            if content:
                print(f"  [CHUNK] '{content}'")
        elif kind == "on_chat_model_end":
            print(f"  [END] Генерация завершена")


asyncio.run(stream_with_events())


ПРИМЕР 8: Стриминг событий


RuntimeError: asyncio.run() cannot be called from a running event loop

# ============================================================================
# ЧАСТЬ 4: БАТЧИНГ И КОНКУРЕНТНОСТЬ
# ============================================================================

In [16]:
"""
Пакетная обработка с batch().
"""
print("\n" + "=" * 60)
print("ПРИМЕР 9: Пакетная обработка")
print("=" * 60)

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)
prompt = ChatPromptTemplate.from_template("Переведи на английский: {text}")
chain = prompt | model | StrOutputParser()

texts = [{"text": "Привет"}, {"text": "Мир"}, {"text": "Python"}, {"text": "Программирование"}]

print("\n9.1. Batch без ограничения конкурентности:")
results = chain.batch(texts)
for inp, out in zip(texts, results):
    print(f"  '{inp['text']}' -> '{out}'")



ПРИМЕР 9: Пакетная обработка

9.1. Batch без ограничения конкурентности:
  'Привет' -> 'Hello.'
  'Мир' -> 'The translation of "Мир" to English is "World."'
  'Python' -> '"Python" на английском языке остается "Python". Это название языка программирования и не требует перевода.'
  'Программирование' -> 'Programming'


In [17]:
"""
Пакетная обработка с ограничением конкурентности.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 10: Batch с ограничением конкурентности")
print("=" * 60)

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)
prompt = ChatPromptTemplate.from_template("Скажи '{word}' по-французски")
chain = prompt | model | StrOutputParser()

words = [{"word": w} for w in ["привет", "спасибо", "пока", "да", "нет"]]

print("\n10.1. Batch с max_concurrency=2:")
import time

start = time.time()
results = chain.batch(words, config={"max_concurrency": 2})
elapsed = time.time() - start

for inp, out in zip(words, results):
    print(f"  '{inp['word']}' -> '{out}'")
print(f"\n  Время выполнения: {elapsed:.2f}s")


ПРИМЕР 10: Batch с ограничением конкурентности

10.1. Batch с max_concurrency=2:
  'привет' -> 'Привет по-французски будет "Bonjour".'
  'спасибо' -> 'По-французски "спасибо" будет "merci".'
  'пока' -> 'По-французски "пока" будет "au revoir".'
  'да' -> ''Да' по-французски будет 'oui'.'
  'нет' -> ''Нет' по-французски будет 'non'.'

  Время выполнения: 2.20s


In [18]:
"""
Параллельное выполнение через RunnableParallel.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 11: Параллельное выполнение")
print("=" * 60)

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)

# Три независимых анализа
parallel = RunnableParallel(
    {
        "summary": ChatPromptTemplate.from_template("Суммаризируй в одном предложении: {text}")
        | model
        | StrOutputParser(),
        "language": ChatPromptTemplate.from_template("На каком языке написан текст? Ответь одним словом: {text}")
        | model
        | StrOutputParser(),
        "word_count": RunnableLambda(lambda x: len(x["text"].split())),
    }
)

text = "Python is a versatile programming language used for web development, data science, and automation."

print("\n11.1. Параллельный анализ текста:")

import time

start = time.time()
result = parallel.invoke({"text": text})
elapsed = time.time() - start

print(f"  Summary: {result['summary']}")
print(f"  Language: {result['language']}")
print(f"  Word count: {result['word_count']}")
print(f"  Время: {elapsed:.2f}s")


ПРИМЕР 11: Параллельное выполнение

11.1. Параллельный анализ текста:
  Summary: Python is a versatile programming language utilized in web development, data science, and automation.
  Language: Английский.
  Word count: 14
  Время: 1.79s


# ============================================================================
# ЧАСТЬ 5: RUNTIME-КОНФИГУРАЦИЯ
# ============================================================================

In [ ]:
"""
ConfigurableField — параметризуемые компоненты.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 12: ConfigurableField")
print("=" * 60)

from langchain_core.runnables import ConfigurableField

# Модель с настраиваемой temperature
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0.5,
).configurable_fields(
    temperature=ConfigurableField(id="temperature", name="Температура", description="Контроль креативности (0-1)")
)

prompt = ChatPromptTemplate.from_template("Придумай название для {product}")
chain = prompt | model | StrOutputParser()

print("\n12.1. Разные значения temperature:")

# Низкая температура — более предсказуемо
result1 = chain.invoke({"product": "приложение для медитации"}, config={"configurable": {"temperature": 0.1}})
print(f"  temperature=0.1: {result1}")

# Высокая температура — более креативно
result2 = chain.invoke({"product": "приложение для медитации"}, config={"configurable": {"temperature": 0.9}})
print(f"  temperature=0.9: {result2}")



ПРИМЕР 12: ConfigurableField

12.1. Разные значения temperature:
  temperature=0.1: Вот несколько идей для названия приложения для медитации:

1. **Спокойствие**
2. **Медитативный Путь**
3. **Тишина Ума**
4. **Гармония**
5. **Дыхание Души**
6. **Мгновения Покой**
7. **Свет Внутри**
8. **Звуки Природы**
9. **Мир Внутри**
10. **Состояние Потока**

Надеюсь, одно из этих названий вдохновит вас!
  temperature=0.9: Вот несколько вариантов названий для приложения для медитации:

1. **Тихий Ум**
2. **Момент Покоя**
3. **Свет Внутри**
4. **Гармония Духа**
5. **Mindful Journey**
6. **Звуки Спокойствия**
7. **Эхо Тишины**
8. **Дыхание Природы**
9. **Сердце Медитации**
10. **Мир Внутри**

Надеюсь, одно из этих названий вдохновит вас!


In [22]:
"""
ConfigurableAlternatives — переключение между компонентами.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 13: ConfigurableAlternatives")
print("=" * 60)


from langchain_core.runnables import ConfigurableField


# Базовая модель с альтернативами
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
).configurable_alternatives(
    ConfigurableField(id="model_variant"),
    default_key="default",
    creative=ChatOpenAI(
        base_url=os.getenv("OPENROUTER_BASE_URL"),
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model="gpt-4o-mini",
        temperature=0.9,
    ),
    precise=ChatOpenAI(
        base_url=os.getenv("OPENROUTER_BASE_URL"),
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model="gpt-4o-mini",
        temperature=0,
    ),
)

prompt = ChatPromptTemplate.from_template("Опиши {object} одним предложением")
chain = prompt | model | StrOutputParser()

print("\n13.1. Переключение между вариантами модели:")


result_default = chain.invoke({"object": "кофе"})
print(f"  default: {result_default}")


result_creative = chain.invoke({"object": "кофе"}, config={"configurable": {"model_variant": "creative"}})
print(f"  creative: {result_creative}")


result_precise = chain.invoke({"object": "кофе"}, config={"configurable": {"model_variant": "precise"}})
print(f"  precise: {result_precise}")


ПРИМЕР 13: ConfigurableAlternatives

13.1. Переключение между вариантами модели:
  default: Кофе — это ароматный напиток, получаемый из обжаренных и молотых зерен кофейного дерева, который бодрит и наполняет энергией.
  creative: Кофе — это ароматный напиток, получаемый из обжаренных и смолотых зерен кофе, который дарит бодрость и уют, объединяя людей в непринужденной атмосфере.
  precise: Кофе — это ароматный напиток, получаемый из обжаренных и молотых зерен кофейного дерева, который бодрит и наполняет энергией.


# ============================================================================
# ЧАСТЬ 6: CALLBACKS И НАБЛЮДАЕМОСТЬ
# ============================================================================

In [23]:
"""
Кастомный callback handler для мониторинга.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 14: Кастомный Callback Handler")
print("=" * 60)


from langchain_core.callbacks import BaseCallbackHandler
from datetime import datetime


class MetricsCallback(BaseCallbackHandler):
    """Callback для сбора метрик."""

    def __init__(self):
        self.start_time = None
        self.end_time = None
        self.tokens_used = 0
        self.calls_count = 0

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.start_time = datetime.now()
        self.calls_count += 1
        print(f"  [METRICS] LLM call #{self.calls_count} started")

    def on_llm_end(self, response, **kwargs):
        self.end_time = datetime.now()
        duration = (self.end_time - self.start_time).total_seconds()
        if hasattr(response, "llm_output") and response.llm_output:
            usage = response.llm_output.get("token_usage", {})
            self.tokens_used += usage.get("total_tokens", 0)
        print(f"  [METRICS] LLM call completed in {duration:.2f}s")

    def on_llm_error(self, error, **kwargs):
        print(f"  [METRICS] LLM error: {error}")

    def get_summary(self):
        return {"total_calls": self.calls_count, "total_tokens": self.tokens_used}


model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)
prompt = ChatPromptTemplate.from_template("Что такое {topic}?")
chain = prompt | model | StrOutputParser()

print("\n14.1. Выполнение с метриками:")
metrics = MetricsCallback()

result = chain.invoke({"topic": "нейронные сети"}, config={"callbacks": [metrics]})

print(f"\n  Результат: {result[:80]}...")
print(f"  Метрики: {metrics.get_summary()}")


ПРИМЕР 14: Кастомный Callback Handler

14.1. Выполнение с метриками:
  [METRICS] LLM call #1 started
  [METRICS] LLM call completed in 3.86s

  Результат: Нейронные сети — это класс алгоритмов машинного обучения, вдохновлённый структур...
  Метрики: {'total_calls': 1, 'total_tokens': 342}


In [24]:
"""
Использование нескольких callbacks одновременно.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 15: Множественные callbacks")
print("=" * 60)


from langchain_core.callbacks import BaseCallbackHandler


class LoggingCallback(BaseCallbackHandler):
    def on_llm_start(self, *args, **kwargs):
        print("  [LOG] LLM started")

    def on_llm_end(self, *args, **kwargs):
        print("  [LOG] LLM ended")


class TimingCallback(BaseCallbackHandler):
    def __init__(self):
        self.start = None

    def on_llm_start(self, *args, **kwargs):
        import time

        self.start = time.time()

    def on_llm_end(self, *args, **kwargs):
        import time

        print(f"  [TIME] Duration: {time.time() - self.start:.2f}s")


model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)
prompt = ChatPromptTemplate.from_template("Скажи привет на {language}")
chain = prompt | model | StrOutputParser()

print("\n15.1. Выполнение с двумя callbacks:")
result = chain.invoke({"language": "японском"}, config={"callbacks": [LoggingCallback(), TimingCallback()]})
print(f"  Результат: {result}")


ПРИМЕР 15: Множественные callbacks

15.1. Выполнение с двумя callbacks:
  [LOG] LLM started
  [LOG] LLM ended
  [TIME] Duration: 1.00s
  Результат: Привет на японском языке будет "こんにちは" (конничива).


# ============================================================================
# ЧАСТЬ 7: КЭШИРОВАНИЕ
# ============================================================================

In [31]:
"""
InMemoryCache — кэширование в памяти.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 16: InMemoryCache")
print("=" * 60)


from langchain_community.cache import InMemoryCache
from langchain_core.globals import set_llm_cache
import time


# Устанавливаем кэш
set_llm_cache(InMemoryCache())

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)

print("\n16.1. Первый вызов (без кэша):")
start = time.time()
result1 = model.invoke("Что такое Python?")
print(f"  Время: {time.time() - start:.2f}s")
print(f"  Ответ: {result1.content[:60]}...")

print("\n16.2. Второй вызов (из кэша):")
start = time.time()
result2 = model.invoke("Что такое Python?")
print(f"  Время: {time.time() - start:.4f}s")
print(f"  Ответ: {result2.content[:60]}...")

# Очищаем кэш
set_llm_cache(None)


ПРИМЕР 16: InMemoryCache

16.1. Первый вызов (без кэша):
  Время: 4.92s
  Ответ: Python — это высокоуровневый язык программирования, который ...

16.2. Второй вызов (из кэша):
  Время: 0.0003s
  Ответ: Python — это высокоуровневый язык программирования, который ...


In [34]:
"""
SQLiteCache — персистентное кэширование.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 17: SQLiteCache")
print("=" * 60)

from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache
import time
import os

cache_path = "2_2_langchain_cache.db"

# Удаляем старый кэш если есть
if os.path.exists(cache_path):
    os.remove(cache_path)

# Устанавливаем SQLite кэш
set_llm_cache(SQLiteCache(database_path=cache_path))

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)

print(f"\n17.1. Кэш сохраняется в: {cache_path}")

print("\n17.2. Первый вызов:")
start = time.time()
result1 = model.invoke("Объясни что такое API")
print(f"  Время: {time.time() - start:.2f}s")

print("\n17.3. Повторный вызов:")
start = time.time()
result2 = model.invoke("Объясни что такое API")
print(f"  Время: {time.time() - start:.4f}s")

# Очищаем кэш
set_llm_cache(None)

# Удаляем файл кэша
# if os.path.exists(cache_path):
#     os.remove(cache_path)


ПРИМЕР 17: SQLiteCache

17.1. Кэш сохраняется в: langchain_cache.db

17.2. Первый вызов:
  Время: 3.98s

17.3. Повторный вызов:
  Время: 0.0008s
